# 84 — Shared M2 registry (structural + proof keys)

- structural key = mathematical M2 problem
- proof key = schema + source + notebook (no code-drift reuse)
- CAND-000005 backfill defaults to **LEGACY_QUARANTINE** (not STRICT)
- package-local audits under `audits/m2_shared_parent.json`


In [ ]:
from pathlib import Path
import os, sys, json
BUNDLE_NAME = 'validated_4d_su2_rg_codex_bundle'
explicit = os.environ.get('VALIDATED_RG_PROJECT_ROOT')
candidates = ([Path(explicit)] if explicit else []) + [
    Path.cwd(), Path.cwd().parent, Path.cwd() / BUNDLE_NAME,
    Path('/notebooks') / BUNDLE_NAME, Path('/notebooks'),
]
PROJECT_ROOT = next((
    p.expanduser().resolve()
    for p in candidates
    if (p.expanduser() / 'src' / 'm2_shared_registry.py').is_file()
), None)
if PROJECT_ROOT is None:
    raise RuntimeError('Pull latest main; src/m2_shared_registry.py not found.')
PERSIST_ROOT = Path(os.environ.get('VALIDATED_RG_PERSIST_ROOT', '/storage/validated_4d_su2_rg')).expanduser().resolve()
os.environ['VALIDATED_RG_PROJECT_ROOT'] = str(PROJECT_ROOT)
os.environ['VALIDATED_RG_PERSIST_ROOT'] = str(PERSIST_ROOT)
os.environ.setdefault('VALIDATED_RG_M7C_RUN_ID', 'M7-20260720T081500Z-c8d5f02b3c96')
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
print('PROJECT_ROOT', PROJECT_ROOT)


In [ ]:
from src.m2_compatibility import keys_from_project
from src.common import hash_tree
from src.m7_status import M7_RUN_ID_CAMPAIGN_C

M7C = os.environ.get('VALIDATED_RG_M7C_RUN_ID', M7_RUN_ID_CAMPAIGN_C)
keys = keys_from_project(
    PROJECT_ROOT,
    j2_max=2,
    source_hash=hash_tree(PROJECT_ROOT / 'src'),
    notebook_hash='84_shared_m2_registry.ipynb',
)
print(json.dumps({
    'structural_key': keys['structural_key'][:16] + '...',
    'proof_key': keys['proof_key'][:16] + '...',
    'shared_run_id': keys['shared_run_id'],
}, indent=2))


## Legacy backfill (CAND-000005) — quarantine until reverify


In [ ]:
from src.m2_shared_registry import MODE_LEGACY, backfill_shared_m2_from_package
from src.m7_candidate_queue import package_root_for, search_root_for

SEARCH = search_root_for(PERSIST_ROOT, M7C)
PKG = package_root_for(SEARCH, 'CAND-000005-ac85c5e9ba38')
if not (PKG / 'child_run_ids.json').is_file():
    raise FileNotFoundError(PKG)
result = backfill_shared_m2_from_package(
    PERSIST_ROOT, PKG,
    project_root=PROJECT_ROOT,
    registration_mode=MODE_LEGACY,
)
print(json.dumps({
    'registry_state': result['record'].get('registry_state'),
    'canonical_run_id': result['record'].get('canonical_run_id'),
    'binding_state': result['binding'].get('state'),
    'note': result['binding'].get('note'),
}, indent=2, ensure_ascii=False))
print('STRICT production reuse requires reverify or a new canonical shared M2.')
